# Stage 9b — Exhaustion test (exit-side, credit spreads)

Different question from `reversion_test.py`: that test asks whether an
already-aged trend predicts reversal AT ENTRY (answer: no — momentum,
~13-15% longer, see README). This asks, for an already-OPEN position:
once a trend crosses its own median / 75th / 90th percentile duration,
how much MORE time typically remains, and is that remaining time tightly
clustered enough to act on as a real profit-booking signal?

Not an entry signal — must not be confused with or override
`trend_age_note`. Pure duration data only, same family as the existing
reversion test, no new indicators.

Design notes (see `src/exhaustion_test.py` docstring for full detail):
- pools excess as a **ratio** of the checkpoint (excess_days / checkpoint_days),
  not raw days — mixing a 10-day-median stock with a 60-day-median stock in
  raw days would misread pure scale difference as scatter
- checkpoints are **nested** — a p90 crossing has already crossed p75 and
  median too, not three independent samples
- median checkpoint overlaps `reversion_test.py`'s existing finding —
  what's actually new here is p75/p90 plus the **spread** of the excess,
  not just its average
- caveat: each stock's own p90 duration rests on a small per-stock sample
  (same >=15 closed-flips floor) — a handful of tail observations, same
  caution as the debit spread scorer's outlier-reliance flag

In [1]:
import sys
sys.path.append("..")

from src.instruments import get_fno_stock_list
from src.auth import get_kite
from src.exhaustion_test import run_exhaustion_test

kite = get_kite()
stock_list = get_fno_stock_list(kite)
all_symbols = stock_list + ["NIFTY", "BANKNIFTY", "SENSEX", "NIFTYNXT50", "MIDCPNIFTY", "FINNIFTY"]
all_symbols = list(dict.fromkeys(all_symbols))

per_stock, universe = run_exhaustion_test(all_symbols)
print(f"Universe: {len(all_symbols)} instruments, per-stock rows: {len(per_stock)}")

Universe: 214 instruments, per-stock rows: 642


In [2]:
universe

,checkpoint,total_crossings_pooled,instruments_contributing,mean_excess_ratio,median_excess_ratio,std_excess_ratio,p25_excess_ratio,p75_excess_ratio,pct_ended_within_25pct_more,pct_ended_within_50pct_more,pct_ended_within_75pct_more,pct_ended_within_100pct_more
0,median,6681,214,1.13,0.78,1.14,0.32,1.54,19.9,35.8,48.8,59.8
1,p75,3390,214,0.63,0.42,0.65,0.17,0.86,34.4,56.6,70.5,80.3
2,p90,1423,214,0.40,0.27,0.42,0.09,0.58,48.3,70.3,82.4,91.4


In [3]:
# Coefficient of variation (std/mean) per checkpoint - the real test of
# "does uncertainty tighten the deeper into the tail you go"
for _, r in universe.iterrows():
    cv = round(r["std_excess_ratio"] / r["mean_excess_ratio"], 2)
    print(f"{r['checkpoint']}: mean={r['mean_excess_ratio']} std={r['std_excess_ratio']} "
          f"cv={cv} | ended within 100% more time: {r['pct_ended_within_100pct_more']}%")

median: mean=1.13 std=1.14 cv=1.01 | ended within 100% more time: 59.8%
p75: mean=0.63 std=0.65 cv=1.03 | ended within 100% more time: 80.3%
p90: mean=0.4 std=0.42 cv=1.05 | ended within 100% more time: 91.4%


## Verdict: no usable exhaustion signal at any checkpoint

The coefficient of variation (std/mean) sits at **~1.0-1.03 at all three
checkpoints** — meaning the spread of "how much longer it ran" is roughly
as large as the average itself, at every checkpoint, with no tightening
the further into the tail you go. That's the signature of a roughly
memoryless process: knowing a trend has already run past a checkpoint
tells you almost nothing precise about how much longer it has left, in
relative terms.

Concretely: even at the most extreme checkpoint (p90), 91.3% of crossings
still needed anywhere up to *double* the p90 duration again before
actually ending — too wide a window to act on as a profit-booking trigger.
At the median checkpoint, only 59.8% had ended even after doubling the
median again.

**Conclusion: pure duration data does not support a reliable exit-side
exhaustion signal, at the median, 75th, or 90th percentile checkpoint.**
This is consistent with (not contradicting) the entry-side finding —
both point the same direction: duration alone behaves like momentum
without a predictable expiry, not like a mean-reverting clock. Per scope,
no live-check function is being built, since none of the three checkpoints
cleared the bar for "tight enough to act on." Keep managing exits by
price action / manual judgment as before — this test's honest negative
result is the useful output here, not a signal forced where none exists.